## Section 0, import

In [ ]:
#Imports
import os
import subprocess
import time
import requests
import h5py
import json
import numpy as np
import re
from pathlib import Path
from datetime import datetime

import DM as dm

## Section 1, user configuration

In [ ]:
#Edit these variables to elabFTW access and data folder
AUTH = "" # Univocal authentication number of the user
ELAB_URL = "" # Experimental session URL
EXP_NUMBER = "" # Experimental session number
EXP_URL = ELAB_URL+EXP_NUMBER

INPUT_FOLDER = ""
SAMPLE_NAME = ""
OUTFILE_NAME = "TEM-STEM_CNR-IMM@CT.nxs"
OUTFILE = INPUT_FOLDER+OUTFILE_NAME

session = requests.Session()
session.trust_env = False

r=session.get(EXP_URL, headers={"Authorization":AUTH}, verify=False)
elab = json.loads(r.text)
elab

c:\Users\hp\anaconda3\envs\py3.11\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '150.145.42.140'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'access_key': None,
 'body': '<p>Template to be completed before a session at the FIB-SEM or TEM microscope. The information required concerns the operator and the sample analysed</p>',
 'body_html': '<p>Template to be completed before a session at the FIB-SEM or TEM microscope. The information required concerns the operator and the sample analysed</p>',
 'canread': '{"base": 30, "teams": [], "users": [], "teamgroups": []}',
 'canread_is_immutable': 0,
 'canwrite': '{"base": 20, "teams": [], "users": [], "teamgroups": []}',
 'canwrite_is_immutable': 0,
 'category': None,
 'category_color': None,
 'category_title': None,
 'comments': [],
 'compounds': [],
 'containers': [],
 'content_type': 1,
 'created_at': '2026-03-12 10:46:23',
 'custom_id': None,
 'date': '2026-03-12',
 'elabid': '20260312-5723a776b87dd9f1e255e201e5a28597b755ce47',
 'events_start': None,
 'events_start_itemid': None,
 'exclusive_edit_mode': [],
 'experiments_links': [],
 'firstname': 'Elisa',
 'fullname': 'Elisa Br

## Section 2, read and list TEM/STEM session files

In [ ]:
# Filter .dm4 files
file_dm4 = [f for f in Path(INPUT_FOLDER).iterdir() if f.is_file() and f.suffix.lower() == '.dm4']

# Ordered files
ordered_files = sorted(file_dm4, key=lambda f: f.stat().st_ctime)

# Stamp
print("Ordered files:")
for f in ordered_files:
    data_creazione = datetime.fromtimestamp(f.stat().st_ctime)
    print(f"{f.name} - {data_creazione.strftime('%Y-%m-%d %H:%M:%S')}")
    print(str(f))

File ordinati:
200000.0V_1500X_0309.dm4 - 2025-07-24 15:09:15
C:\Users\hp\Desktop\CNR-IMM\NFFA\DATI\TEM\prova\200000.0V_1500X_0309.dm4
200000.0V_80000X_0301.dm4 - 2025-07-24 15:09:23
C:\Users\hp\Desktop\CNR-IMM\NFFA\DATI\TEM\prova\200000.0V_80000X_0301.dm4
200000.0V_1500000X_0315.dm4 - 2025-07-24 15:10:12
C:\Users\hp\Desktop\CNR-IMM\NFFA\DATI\TEM\prova\200000.0V_1500000X_0315.dm4


## Section 3, write into a NeXus file

In [ ]:
with h5py.File(OUTFILE, "w") as f:
    entry = f.create_group("ENTRY")
    entry.attrs["NX_class"] = "NXentry"
    entry.create_dataset("definition", data="NXem")
    entry.create_dataset("identifier_experiment", data = elab['metadata_decoded']['extra_fields']['Sample history']['value'])
    entry.create_dataset("start_time", data=elab['date'])
    entry["definition"].attrs["version"] = "v2025.11"

    # USER INFO 
    user = entry.create_group("userID")
    user.attrs["NX_class"] = "NXuser"
    user.create_dataset("name", data=elab['fullname'])
    user.create_dataset("affiliation", data=elab['metadata_decoded']['extra_fields']['Affiliation']['value'])
    user.create_dataset("address", data=elab['metadata_decoded']['extra_fields']['Office address']['value'])
    user.create_dataset("email", data=elab['metadata_decoded']['extra_fields']['Operator email']['value'])
    user.create_dataset("orcid", data=elab['orcid'])

    # SAMPLE INFO
    sample = entry.create_group("sampleID")
    sample.attrs["NX_class"] = "NXsample"
    sample.create_dataset("physiscal_form",data= elab["metadata_decoded"]['extra_fields']['Physical form']['value'])
    issim = elab['metadata_decoded']['extra_fields']['Method']['value'] != "experiment"
    sample.create_dataset("is_simulation", data=issim)
    sample.create_dataset("name", data = elab['metadata_decoded']['extra_fields']['Sample name']['value'])
    sample.create_dataset("identifier_sample", data = elab['metadata_decoded']['extra_fields']['Sample name']['value'])
    sample.create_dataset("identifier_parent", data = elab['metadata_decoded']['extra_fields']['Sample history']['value'])
    sample.create_dataset("preparation_date", data = elab['metadata_decoded']['extra_fields']['Preparation date']['value'])
    sample.create_dataset("atom_types", data = elab['metadata_decoded']['extra_fields']['Atom types']['value'])

    # MYCROSCOPE AND SOFTWARE INFO

    image,dimensions,calibration,metadata =dm.dm_load(str(ordered_files[0]))
    
    meas = entry.create_group("measurement")
    meas.attrs["NX_class"] = "NXem_measurement" 

    em = meas.create_group("instrument")
    em.attrs["NXclass"] = "NXem_instrument"
    em.create_dataset("instrument_name", data=elab['metadata_decoded']['extra_fields']['Electron Microscope Used']['value'])
    em.create_dataset("location", data = "CNR-IMM Catania")
    em.create_dataset("type", data = "TEM/STEM")

    fab = em.create_group("FABRICATION")
    fab.attrs["NXclass"] = "NXfabrication"
    fab.create_dataset("vendor", data = "Jeol")
    fab.create_dataset("model", data = "JEM 2010F")
    fab.create_dataset("identifier", data = "")

    software = em.create_group("programID")
    software.attrs["NX_class"] = "NXprogram"
    software.create_dataset("program", data = metadata['Acquisition']['Device']['Name'])
    
    ebeam = em.create_group("ebeam_column")
    ebeam.attrs["NXclass"] = "NXebeam_column"
    esource = ebeam.create_group("electron_source")
    esource.attrs["NXclass"] = "NXsource"
    esource.create_dataset("name", data = "")
    esource.create_dataset("emitter_type", data = "Schottky FEG")

    # LABORATORY SESSION CREATION
    event = meas.create_group("eventID")
    event.attrs["NX_class"] = "NXem_event_data"
    splitted_data = elab['metadata_decoded']['extra_fields']['Analysis date']['value'].split(sep='-')
    updated_data = str(splitted_data[2])+'/'+str(splitted_data[1])+'/'+str(splitted_data[0])
    event.create_dataset("start_time", data=updated_data)

    # MULTIPLE DATA DURING LABORATORY SESSION
    for idx, tif_path in enumerate(ordered_files):
        img_name = str(tif_path)
        img_data,dimensions,calibration,img_meta =dm.dm_load(img_name)
        image = event.create_group(f"image_{idx+1}")
        image.attrs["NX_class"] = "NXimage"
        image_2d = image.create_group("image_2d")
        image_2d.attrs["NX_class"] = "NXdata"
        image_2d.create_dataset("title", data=img_meta['Microscope Info']['Illumination Mode']+' '+img_meta['Microscope Info']['Imaging Mode'])
        image_2d.create_dataset(img_meta['Microscope Info']['Illumination Mode']+' '+img_meta['Microscope Info']['Imaging Mode'], data=img_data)

        conf=image.create_group("instrument")
        conf.attrs["NX_class"] = "NXem_instrument"

        colset = conf.create_group("ebeam_column")
        colset.attrs["NX_class"] = "NXebeam_column"
        colset.create_dataset("operation_mode", data = img_meta['Microscope Info']['Operation Mode'])

        elsource = colset.create_group("electron_source")
        elsource.attrs["NX_class"] = "NXsource"
        elsource.create_dataset("voltage", data = float(img_meta['Microscope Info']['Voltage']))
        elsource["voltage"].attrs["units"] = "V"
        elsource.create_dataset("emission_current", data = float(img_meta['Microscope Info']['Emission Current (µA)']))
        elsource["emission_current"].attrs["units"] = "microA"

        optics = conf.create_group("optics")
        optics.attrs["NX_class"] = "NXem_optical_system"
        optics.create_dataset("working_distance", data= float(img_meta['Microscope Info']['STEM Camera Length']))
        optics["working_distance"].attrs["units"] = ""
        optics.create_dataset("magnification", data = float(img_meta['Microscope Info']['Indicated Magnification']))


## Section 4, validation of the output file

In [ ]:
##!punx validate --report ERROR {OUTFILE}
!punx tree {OUTFILE}

root:NXroot
  ENTRY:NXentry
    definition = 'NXem'
      @version = 'v2024.02'
    identifier_experiment = '*'
    measurement:NXem_measurement
      eventID:NXem_event_data
        image_1:NXimage
          image_2d:NXdata
            imag = uint8(1094x1536)
            title = 'EBeam TLD SE'
          instrument:NXem_instrument
            detectorID:NXdetector
              operation_mode = 'TLD SE'
            ebeam_column:NXebeam_column
              electron_source:NXsource
                emission_current = 4e-10
                  @units = 'A'
                voltage = 2000.0
                  @units = 'V'
              operation_mode = 'TLD SE'
              scan_controller:NXscan_controller
                dwell_time = 1.5e-05
                scan_schema = 'PIA 4'
            stageID:NXmanipulator
              position = [-0.007252  0.012726  0.003999]
                @units = 'm,m,m'
              rotation = -0.544151
                @units = 'degree'
              tilt1 = 